# 05 · 記憶與狀態管理

到目前為止 agent 是**金魚腦**：每問一題就開一段新對話，答完即忘。這課給它記憶。

- **短期記憶**：把每回合訊息累積在 list，每次都帶上 → 記得前面說過什麼。
- **Scratchpad**：ReAct 的 Thought/Action/Observation，就是解任務的草稿紙。
- **長期記憶**：對話太長會撐爆 context window → 請模型把舊對話**摘要**，騰出空間。

## 1. 載入模型

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 2. 一個 `Memory` 類別

累積對話；當太長時，把較舊的對話交給模型**摘要**，用摘要換掉逐字歷史。

In [ ]:
class Memory:
    def __init__(self, system, max_turns=6):
        self.system = system
        self.history = []          # [{"role","content"}, ...]
        self.summary = ""
        self.max_turns = max_turns

    def add(self, role, content):
        self.history.append({"role": role, "content": content})

    def messages(self):
        sys = self.system
        if self.summary:
            sys += "\n\n先前對話摘要：" + self.summary
        return [{"role": "system", "content": sys}] + self.history

    def maybe_summarize(self):
        if len(self.history) <= self.max_turns:
            return
        old, keep = self.history[:-2], self.history[-2:]   # 最近一回合保留逐字
        text = "\n".join(f'{m["role"]}: {m["content"]}' for m in old)
        s = chat([{"role": "user",
                   "content": "用三句話摘要對話重點與已知事實：\n" + text}],
                 max_new_tokens=200)
        self.summary = (self.summary + " " + s).strip()
        self.history = keep
        print("  〔已摘要舊對話 → 騰出 context〕")

## 3. 多回合對話：它記得住嗎？

In [ ]:
mem = Memory("你是友善的繁體中文助手。", max_turns=4)


def say(u):
    mem.add("user", u)
    r = chat(mem.messages())
    mem.add("assistant", r)
    mem.maybe_summarize()
    print("你：", u, "\n助手：", r, "\n")


say("我叫小明，最喜歡爬山。")
say("根據我的喜好，推薦一個週末活動。")   # 應該扣到「爬山」
say("我剛剛說我叫什麼名字？")             # 測短期記憶

## 小結

- **短期記憶**＝累積 messages；**scratchpad**＝任務草稿；**長期記憶**＝摘要壓縮。
- **context window 有限**，主動摘要是長對話 agent 撐得久的祕訣。
- 摘要會遺失細節——這是「記得久」與「記得準」的取捨。

下一課：給 agent **RAG**，讓它查得到它沒學過的事實。